In [1]:
import pandas as pd
import numpy as np
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output

# Generate simulated data with the exact same structure
np.random.seed(42)
years = np.arange(1980, 2014)
# Changed 'ME' to 'M' for compatibility with older pandas versions
dates = pd.date_range(start='1980-01-01', end='2013-12-31', freq='M') 
recession = [1 if (y in [1981, 1982, 1991, 2008, 2009]) else 0 for y in dates.year]
vehicle_types = ['Supermini', 'SmallFamily', 'Medium', 'Large', 'Executive']

data = []
for d, r in zip(dates, recession):
    for v in vehicle_types:
        data.append({
            'Date': d,
            'Recession': r,
            'Automobile_Sales': np.random.randint(1000, 5000) if r == 0 else np.random.randint(500, 2000),
            'GDP': np.random.uniform(5000, 20000),
            'unemployment_rate': np.random.uniform(4, 12),
            'Consumer_Confidence': np.random.uniform(50, 100),
            'Seasonality_Weight': np.random.uniform(0.5, 1.5),
            'Price': np.random.uniform(10000, 50000),
            'Advertising_Expenditure': np.random.uniform(1000, 10000),
            'Vehicle_Type': v,
            'Competition': np.random.randint(1, 10)
        })

df = pd.DataFrame(data)
df['Month'] = df['Date'].dt.month_name()
df['Year'] = df['Date'].dt.year

app = dash.Dash(__name__)
app.config.suppress_callback_exceptions = True

print("Simulated Data generated successfully!")

Simulated Data generated successfully!


In [2]:
title = html.H1('Automobile Sales Statistics Dashboard', 
                style={'textAlign': 'center', 'color': '#503D36', 'font-size': 24})

In [3]:
dropdown_stat = html.Div([
    html.Label("Select Statistics:"),
    dcc.Dropdown(
        id='dropdown-statistics',
        options=[
            {'label': 'Yearly Statistics', 'value': 'Yearly Statistics'},
            {'label': 'Recession Period Statistics', 'value': 'Recession Period Statistics'}
        ],
        value='Yearly Statistics',
        placeholder='Select a report type',
        style={'width': '80%', 'padding': '3px', 'font-size': '20px', 'text-align-last': 'center'}
    )
])

In [4]:
dropdown_year = html.Div([
    html.Label("Select Year:"),
    dcc.Dropdown(
        id='select-year',
        options=[{'label': i, 'value': i} for i in range(1980, 2014)],
        value=1980,
        placeholder='Select a year',
        style={'width': '80%', 'padding': '3px', 'font-size': '20px', 'text-align-last': 'center'}
    )
])

In [5]:
output_box = html.Div(id='output-container', 
                      className='chart-grid', 
                      style={'display': 'flex', 'flex-direction': 'column'})

In [6]:
app.layout = html.Div([
    title,
    dropdown_stat,
    dropdown_year,
    output_box
])

In [7]:
@app.callback(
    Output(component_id='select-year', component_property='disabled'),
    Input(component_id='dropdown-statistics', component_property='value')
)
def update_input_container(selected_statistics):
    if selected_statistics == 'Yearly Statistics':
        return False
    else:
        return True

In [8]:
import plotly.express as px

def create_recession_charts():
    rec_data = df[df['Recession'] == 1]
    
    # Chart 1: Line
    yearly_rec = rec_data.groupby('Year')['Automobile_Sales'].mean().reset_index()
    fig1 = px.line(yearly_rec, x='Year', y='Automobile_Sales', title="Average Automobile Sales Fluctuation over Recession Period")
    
    # Chart 2: Bar
    avg_sales = rec_data.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index()
    fig2 = px.bar(avg_sales, x='Vehicle_Type', y='Automobile_Sales', title="Average Number of Vehicles Sold by Vehicle Type")
    
    # Chart 3: Pie
    exp_rec = rec_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index()
    fig3 = px.pie(exp_rec, values='Advertising_Expenditure', names='Vehicle_Type', title="Total Expenditure Share by Vehicle Type during Recessions")
    
    # Chart 4: Bar
    unemp_data = rec_data.groupby(['unemployment_rate', 'Vehicle_Type'])['Automobile_Sales'].mean().reset_index()
    fig4 = px.bar(unemp_data, x='unemployment_rate', y='Automobile_Sales', color='Vehicle_Type', title="Effect of Unemployment Rate on Vehicle Type and Sales")
    
    # Return in 2x2 layout
    return [
        html.Div(children=[dcc.Graph(figure=fig1), dcc.Graph(figure=fig2)], style={'display': 'flex'}),
        html.Div(children=[dcc.Graph(figure=fig3), dcc.Graph(figure=fig4)], style={'display': 'flex'})
    ]

In [9]:
import plotly.express as px

def create_yearly_charts(input_year):
    yearly_data = df[df['Year'] == int(input_year)]
    
    # Chart 1: Line (Overall)
    yas = df.groupby('Year')['Automobile_Sales'].mean().reset_index()
    fig1 = px.line(yas, x='Year', y='Automobile_Sales', title="Total Yearly Automobile Sales")
    
    # Chart 2: Line (Monthly)
    mas = yearly_data.groupby('Month')['Automobile_Sales'].sum().reset_index()
    fig2 = px.line(mas, x='Month', y='Automobile_Sales', title=f"Total Monthly Automobile Sales in {input_year}")
    
    # Chart 3: Bar
    avr_vdata = yearly_data.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index()
    fig3 = px.bar(avr_vdata, x='Vehicle_Type', y='Automobile_Sales', title=f"Average Vehicles Sold by Vehicle Type in {input_year}")
    
    # Chart 4: Pie
    exp_data = yearly_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index()
    fig4 = px.pie(exp_data, values='Advertising_Expenditure', names='Vehicle_Type', title=f"Total Advertisement Expenditure in {input_year}")
    
    return [
        html.Div(children=[dcc.Graph(figure=fig1), dcc.Graph(figure=fig2)], style={'display': 'flex'}),
        html.Div(children=[dcc.Graph(figure=fig3), dcc.Graph(figure=fig4)], style={'display': 'flex'})
    ]

In [10]:
import plotly.express as px

def create_yearly_charts(input_year):
    yearly_data = df[df['Year'] == int(input_year)]
    
    # Chart 1: Line (Overall)
    yas = df.groupby('Year')['Automobile_Sales'].mean().reset_index()
    fig1 = px.line(yas, x='Year', y='Automobile_Sales', title="Total Yearly Automobile Sales")
    
    # Chart 2: Line (Monthly)
    mas = yearly_data.groupby('Month')['Automobile_Sales'].sum().reset_index()
    fig2 = px.line(mas, x='Month', y='Automobile_Sales', title=f"Total Monthly Automobile Sales in {input_year}")
    
    # Chart 3: Bar
    avr_vdata = yearly_data.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index()
    fig3 = px.bar(avr_vdata, x='Vehicle_Type', y='Automobile_Sales', title=f"Average Vehicles Sold by Vehicle Type in {input_year}")
    
    # Chart 4: Pie
    exp_data = yearly_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index()
    fig4 = px.pie(exp_data, values='Advertising_Expenditure', names='Vehicle_Type', title=f"Total Advertisement Expenditure in {input_year}")
    
    return [
        html.Div(children=[dcc.Graph(figure=fig1), dcc.Graph(figure=fig2)], style={'display': 'flex'}),
        html.Div(children=[dcc.Graph(figure=fig3), dcc.Graph(figure=fig4)], style={'display': 'flex'})
    ]

In [11]:
@app.callback(
    Output(component_id='output-container', component_property='children'),
    [Input(component_id='dropdown-statistics', component_property='value'),
     Input(component_id='select-year', component_property='value')]
)
def update_output_container(selected_statistics, input_year):
    if selected_statistics == 'Recession Period Statistics':
        return create_recession_charts()
    elif selected_statistics == 'Yearly Statistics':
        return create_yearly_charts(input_year)
    return None

In [12]:
app.run(debug=False, use_reloader=False, jupyter_mode="tab", port=8059)

Dash app running on http://127.0.0.1:8059/


<IPython.core.display.Javascript object>